# Figuras da comparação das fronteiras — cenário aplicado 8D

Este notebook recria as figuras centrais do notebook original e acrescenta diagnósticos adequados à comparação corrigida. Todas as métricas são lidas dos resultados validados do notebook 11; as projeções físicas são reconstruídas a partir do mesmo RSM e da mesma restrição esférica. As figuras são exportadas em PNG (300 dpi) e PDF.

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "axes.titleweight": "bold"})

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
OUT = ROOT / "results" / "applied" / "comparison_8d_corrected"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

MET = pd.read_csv(OUT / "metricas_por_seed.csv")
DET = pd.read_csv(OUT / "metricas_deterministicos.csv")
RED = pd.read_csv(OUT / "reducao_resumo.csv")
RED_SEED = pd.read_csv(OUT / "reducao_por_seed.csv")
TESTS = pd.read_csv(OUT / "testes_wilcoxon_holm.csv")

COLORS = {"VRF-NBI": "#4C72B0", "CNBI": "#DD8452", "NSGA-III": "#55A868", "MOEA/D": "#C44E52", "Referência P*": "#777777"}
METHOD_ORDER = ["VRF-NBI", "CNBI", "NSGA-III", "MOEA/D"]
METRICS = ["GD1", "IGD1", "HV", "Spacing", "Sparsity"]
LABELS = {"GD1":r"$GD_1$ ↓", "IGD1":r"$IGD_1$ ↓", "HV":"HV ↑", "Spacing":"Spacing ↓", "Sparsity":"Sparsity ↓", "n":"Cardinalidade"}

def savefig(fig, stem):
    fig.tight_layout()
    fig.savefig(FIG / f"{stem}.png", bbox_inches="tight", facecolor="white")
    fig.savefig(FIG / f"{stem}.pdf", bbox_inches="tight", facecolor="white")
    plt.close(fig)

print(f"Resultados: {OUT}")
print(f"Figuras: {FIG}")
display(MET.groupby("method")[METRICS + ["n"]].median().round(5))

In [ ]:
# Reconstrução das frentes físicas e normalizadas
DOE = ROOT / 'data' / 'applied' / 'VRF_artigo.xlsx'
VRF_FILE = ROOT / 'data' / 'applied' / 'VRF_Pareto.xlsx'
CNBI_FILE = ROOT / 'data' / 'applied' / 'fronteira_ND_cnbi.csv'
REF_FILE = ROOT / "data" / "reference_fronts" / "referencia_pareto_8D_nsga3_moead.csv"
EA_DIR = ROOT / "results" / "applied" / "applied_8d_equal_budget" / "final_fronts"

XCOLS = ["cs", "f", "md"]
YCOLS = ["T", "MTTF", "WR", "Ra", "Rt", "Kp", "ROI", "OEE"]
SIGNS = np.array([-1, -1, 1, 1, 1, 1, -1, -1], float)
ALPHA = 2**0.75

doe = pd.read_excel(DOE, sheet_name="Plan1")
Xdoe = doe[XCOLS].to_numpy(float)
Ydoe = doe[YCOLS].to_numpy(float)
rsm = Pipeline([("poly", PolynomialFeatures(2, include_bias=False)),
                ("reg", LinearRegression())]).fit(Xdoe, Ydoe)

def predict(X):
    X = np.asarray(X, float)
    return rsm.predict(X)

def sanitize(X):
    X = np.asarray(X, float)
    X = X[np.isfinite(X).all(axis=1)]
    X = X[(np.linalg.norm(X, axis=1) <= ALPHA + 1e-8)]
    X = np.unique(np.round(X, 12), axis=0)
    F = predict(X) * SIGNS
    nd = NonDominatedSorting().do(F, only_non_dominated_front=True)
    return X[nd], predict(X[nd])

def read_x_csv(path):
    d = pd.read_csv(path)
    return d[XCOLS].to_numpy(float)

Xp, Yp = sanitize(read_x_csv(REF_FILE))
Xv, Yv = sanitize(pd.read_excel(VRF_FILE)[XCOLS].to_numpy(float))
Xc, Yc = sanitize(pd.read_csv(CNBI_FILE)[XCOLS].to_numpy(float))

Xe, BEST_EA_SEEDS = {}, {}
for method, prefix in [("NSGA-III", "nsga3"), ("MOEA/D", "moead")]:
    row = MET.loc[MET.method.eq(method)].sort_values(["IGD1", "seed"]).iloc[0]
    best_seed = int(row.seed)
    path = EA_DIR / f"{prefix}_seed{best_seed:03d}.csv"
    Xe[method], _ = sanitize(read_x_csv(path))
    BEST_EA_SEEDS[method] = {"seed": best_seed, "IGD1": float(row.IGD1), "file": path.name}

ideal = (Yp * SIGNS).min(axis=0)
nadir = (Yp * SIGNS).max(axis=0)
amp = np.maximum(nadir - ideal, 1e-12)
norm = lambda Y: ((Y * SIGNS) - ideal) / amp

FRONTS = {"Referência P*": Yp, "VRF-NBI": Yv, "CNBI": Yc,
          "NSGA-III": predict(Xe["NSGA-III"]), "MOEA/D": predict(Xe["MOEA/D"])}
PLOT_LABELS = {m: (f"{m} — melhor $IGD_1$ (seed {BEST_EA_SEEDS[m]['seed']})" if m in BEST_EA_SEEDS else m) for m in FRONTS}
print("Execuções selecionadas:", BEST_EA_SEEDS)
print({k: len(v) for k, v in FRONTS.items()})

In [ ]:
# 1 — Métricas principais (equivalente melhorado ao gráfico de barras original)
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
for ax, metric in zip(axes.flat, METRICS + ["n"]):
    vals, lo, hi = [], [], []
    for method in METHOD_ORDER:
        if method in ("VRF-NBI", "CNBI"):
            row = DET.loc[DET.method.eq(method)].iloc[0]
            v = float(row[metric]); q1 = q3 = v
        else:
            s = MET.loc[MET.method.eq(method), metric]
            v, q1, q3 = s.median(), s.quantile(.25), s.quantile(.75)
        vals.append(v); lo.append(v-q1); hi.append(q3-v)
    x = np.arange(4)
    ax.bar(x, vals, color=[COLORS[m] for m in METHOD_ORDER], alpha=.9)
    ax.errorbar(x, vals, yerr=np.vstack([lo, hi]), fmt="none", ecolor="black", capsize=4, lw=1)
    ax.set_xticks(x, METHOD_ORDER, rotation=18, ha="right")
    ax.set_title(LABELS[metric]); ax.set_ylabel("valor")
fig.suptitle("Qualidade das fronteiras — determinísticos e mediana dos 10 seeds", fontsize=15, fontweight="bold", y=1.01)
savefig(fig, "fig01_metricas_principais")

# 2 — Variabilidade por seed, com referências determinísticas
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
for ax, metric in zip(axes.flat, METRICS + ["n"]):
    sns.boxplot(data=MET, x="method", y=metric, order=["NSGA-III", "MOEA/D"],
                palette=[COLORS["NSGA-III"], COLORS["MOEA/D"]], width=.55, ax=ax)
    sns.stripplot(data=MET, x="method", y=metric, order=["NSGA-III", "MOEA/D"],
                  color="black", alpha=.6, size=4, jitter=.12, ax=ax)
    for dm in ["CNBI", "VRF-NBI"]:
        v = float(DET.loc[DET.method.eq(dm), metric].iloc[0])
        ax.axhline(v, color=COLORS[dm], ls="--", lw=1.4, label=dm)
    ax.set_title(LABELS[metric]); ax.set_xlabel("")
axes.flat[0].legend(frameon=True, fontsize=8)
fig.suptitle("Variabilidade entre sementes e níveis das frentes determinísticas", fontsize=15, fontweight="bold", y=1.01)
savefig(fig, "fig02_variabilidade_seeds")

# 3 — Projeções físicas em pares de objetivos (mesma ideia do original)
pairs = [(0,5), (3,6), (2,7), (1,4)]
rng = np.random.default_rng(1010)
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (i,j) in zip(axes.flat, pairs):
    for method in ["Referência P*", "VRF-NBI", "CNBI", "NSGA-III", "MOEA/D"]:
        Y = FRONTS[method]
        if len(Y) > 7000:
            Y = Y[rng.choice(len(Y), 7000, replace=False)]
        kw = dict(s=9, alpha=.25) if method == "Referência P*" else dict(s=18, alpha=.65)
        ax.scatter(Y[:,i], Y[:,j], color=COLORS[method], label=PLOT_LABELS[method], rasterized=True, **kw)
    ax.set_xlabel(YCOLS[i]); ax.set_ylabel(YCOLS[j]); ax.set_title(f"{YCOLS[i]} × {YCOLS[j]}")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=5, bbox_to_anchor=(.5, -.01))
fig.suptitle("Cobertura das frentes — melhor execução por $IGD_1$", fontsize=15, fontweight="bold")
fig.subplots_adjust(bottom=.10)
savefig(fig, "fig03_projecoes_objetivos")

In [ ]:
# 4 — Métodos de redução para cardinalidade comparável à VRF-NBI
reduction_metrics = METRICS
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, metric in zip(axes.flat, reduction_metrics):
    col = f"{metric}_median"
    d = RED.sort_values(col, ascending=(metric != "HV"))
    ax.barh(d["method"], d[col], color=sns.color_palette("viridis", len(d)))
    vrf = float(DET.loc[DET.method.eq("VRF-NBI"), metric].iloc[0])
    ax.axvline(vrf, color=COLORS["VRF-NBI"], ls="--", lw=2, label="VRF-NBI")
    ax.set_title(LABELS[metric]); ax.invert_yaxis()
axes.flat[-1].axis("off")
axes.flat[0].legend(fontsize=8)
fig.suptitle("Redução da CNBI para a cardinalidade da VRF-NBI", fontsize=15, fontweight="bold", y=1.01)
savefig(fig, "fig04_comparacao_reducoes")

# Seleciona a execução Farthest-point cuja IGD1 é mais próxima da mediana e a reconstrói
fps_rows = RED_SEED.loc[RED_SEED.method.eq("Farthest-point")].copy()
med = fps_rows.IGD1.median()
selected_seed = int(fps_rows.iloc[(fps_rows.IGD1-med).abs().argmin()].seed)
target_n = len(Yv)
Cn = norm(Yc)
def farthest_point_indices(F, k, seed):
    rng = np.random.default_rng(seed)
    chosen = [int(rng.integers(len(F)))]
    mind = cdist(F, F[chosen]).ravel()
    for _ in range(1, min(k, len(F))):
        nxt = int(np.argmax(mind)); chosen.append(nxt)
        mind = np.minimum(mind, cdist(F, F[[nxt]]).ravel())
    return np.array(chosen)
idx = farthest_point_indices(Cn, target_n, selected_seed)
Ycr = Yc[idx]

# 5 — Cobertura VRF-NBI versus CNBI reduzida (equivalente ao original)
fig, axes = plt.subplots(2, 2, figsize=(12, 9.5))
for ax, (i,j) in zip(axes.flat, pairs):
    sample = Yp[rng.choice(len(Yp), min(7000, len(Yp)), replace=False)]
    ax.scatter(sample[:,i], sample[:,j], s=8, alpha=.20, color=COLORS["Referência P*"], label="Referência P*")
    ax.scatter(Yv[:,i], Yv[:,j], s=28, alpha=.80, color=COLORS["VRF-NBI"], label="VRF-NBI")
    ax.scatter(Ycr[:,i], Ycr[:,j], s=28, alpha=.80, color=COLORS["CNBI"], label=f"CNBI reduzida (n={target_n})")
    ax.set_xlabel(YCOLS[i]); ax.set_ylabel(YCOLS[j]); ax.set_title(f"{YCOLS[i]} × {YCOLS[j]}")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(.5, -.01))
fig.suptitle(f"Cobertura com cardinalidade igual — Farthest-point, seed {selected_seed}", fontsize=15, fontweight="bold")
fig.subplots_adjust(bottom=.09)
savefig(fig, "fig05_cobertura_cardinalidade_igual")

In [ ]:
# 6 — Diagnóstico convergência × cobertura
fig, ax = plt.subplots(figsize=(9, 6.5))
for method in ["NSGA-III", "MOEA/D"]:
    d = MET.loc[MET.method.eq(method)]
    sizes = 60 + 260*(d.HV-d.HV.min())/(max(d.HV.max()-d.HV.min(), 1e-12))
    ax.scatter(d.GD1, d.IGD1, s=sizes, color=COLORS[method], alpha=.72, edgecolor="white", label=method)
for method in ["CNBI", "VRF-NBI"]:
    d = DET.loc[DET.method.eq(method)].iloc[0]
    ax.scatter(d.GD1, d.IGD1, marker="*", s=280, color=COLORS[method], edgecolor="black", label=method, zorder=5)
    ax.annotate(method, (d.GD1, d.IGD1), xytext=(6,6), textcoords="offset points", fontsize=9)
ax.set_xlabel(LABELS["GD1"]); ax.set_ylabel(LABELS["IGD1"])
ax.set_title("Convergência × cobertura (tamanho do ponto proporcional ao HV)")
ax.legend(ncol=2)
savefig(fig, "fig06_convergencia_cobertura")

# 7 — Comparação pareada por semente
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8))
for ax, metric in zip(axes, ["GD1", "IGD1", "HV"]):
    wide = MET.pivot(index="seed", columns="method", values=metric)
    for seed, row in wide.iterrows():
        ax.plot([0,1], [row["NSGA-III"], row["MOEA/D"]], color="#999999", alpha=.65, lw=1)
        ax.scatter(0, row["NSGA-III"], color=COLORS["NSGA-III"], s=32)
        ax.scatter(1, row["MOEA/D"], color=COLORS["MOEA/D"], s=32)
    ax.set_xticks([0,1], ["NSGA-III", "MOEA/D"]); ax.set_title(LABELS[metric]); ax.set_ylabel("valor")
fig.suptitle("Diferenças pareadas por semente", fontsize=15, fontweight="bold", y=1.02)
savefig(fig, "fig07_pareamento_sementes")

# 8 — Tamanho de efeito dos testes, com significância após Holm
piv = TESTS.pivot(index="comparison", columns="metric", values="rank_biserial_positive_favors_first").reindex(columns=METRICS)
sig = TESTS.pivot(index="comparison", columns="metric", values="significant_holm_0p05").reindex(columns=METRICS).fillna(False)
annot = piv.copy().astype(object)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.iloc[i,j]
        annot.iloc[i,j] = "" if pd.isna(v) else f"{v:.2f}{'*' if bool(sig.iloc[i,j]) else ''}"
fig, ax = plt.subplots(figsize=(10, 6.8))
sns.heatmap(piv, vmin=-1, vmax=1, center=0, cmap="vlag", annot=annot, fmt="", linewidths=.5,
            cbar_kws={"label":"Correlação bisserial de postos\n(positivo favorece o primeiro método)"}, ax=ax)
ax.set_xlabel(""); ax.set_ylabel(""); ax.set_title("Tamanho de efeito dos testes de Wilcoxon (* Holm p < 0,05)")
savefig(fig, "fig08_efeitos_estatisticos")

# 9 — Perfis medianos no espaço normalizado de minimização
fig, ax = plt.subplots(figsize=(11, 6))
for method in METHOD_ORDER:
    profile = np.median(norm(FRONTS[method]), axis=0)
    ax.plot(YCOLS, profile, marker="o", lw=2.2, color=COLORS[method], label=PLOT_LABELS[method])
ax.axhline(0, color="#777777", lw=.8)
ax.set_ylabel("objetivo normalizado (menor = melhor)")
ax.set_title("Perfil mediano — melhor execução por $IGD_1$ para os EAs")
ax.legend(ncol=2)
savefig(fig, "fig09_perfis_objetivos_normalizados")

manifest = {
    "notebook": "13_figuras_comparacao_fronteiras_8D.ipynb",
    "selected_ea_runs": BEST_EA_SEEDS,
    "selected_reduction": {"method":"Farthest-point", "seed": selected_seed, "target_n": target_n},
    "figures": sorted(p.name for p in FIG.glob("*.png")),
    "formats": ["png (300 dpi)", "pdf"],
}
(FIG / "manifest_figuras.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Geradas {len(manifest['figures'])} figuras em PNG e PDF.")
display(pd.DataFrame({"arquivo": manifest["figures"]}))